# Image Processing API 전처리 테스트

이미지를 하나 로드한 뒤 FastAPI를 통해 `gaussian_blur`, `canny`, `resize`와 기본 `edge_thumbnail` Pipeline을 실행하고 결과를 비교합니다.

먼저 프로젝트 루트의 터미널에서 API 서버를 실행하세요.

```bash
python -m uvicorn src.main:app --host 127.0.0.1 --port 8000
```

Notebook은 다른 터미널에서 실행합니다.

```bash
python -m jupyter lab
```

In [ ]:
from __future__ import annotations

import base64
import json
import mimetypes
from pathlib import Path
from typing import Any

import cv2
import httpx2 as httpx
import matplotlib.pyplot as plt
import numpy as np

BASE_URL = 'http://127.0.0.1:8000'
API_PREFIX = '/api/v1'
REQUEST_TIMEOUT_SECONDS = 60.0

# 자신의 이미지로 테스트할 때 경로를 지정하세요.
# 예: IMAGE_PATH = Path(r'C:/data/sem_image.png')
# None이면 Notebook이 메모리에서 만든 데모 이미지를 사용합니다.
IMAGE_PATH: Path | None = None

# 현재 Canny와 edge_thumbnail Pipeline은 GRAY 입력을 요구합니다.
# Blur/Resize만 컬러로 확인하려면 False로 바꿀 수 있습니다.
FORCE_GRAYSCALE = True

client = httpx.Client(
    base_url=BASE_URL,
    timeout=REQUEST_TIMEOUT_SECONDS,
    trust_env=False,
)

## 1. API 연결 및 등록 항목 확인

서버 상태와 외부에서 조회 가능한 Operation/Pipeline 이름을 확인합니다. 연결 오류가 발생하면 위의 `uvicorn` 명령이 실행 중인지 확인하세요.

In [ ]:
health_response = client.get(f'{API_PREFIX}/health')
health_response.raise_for_status()

operations_response = client.get(f'{API_PREFIX}/operations')
operations_response.raise_for_status()

pipelines_response = client.get(f'{API_PREFIX}/pipelines')
pipelines_response.raise_for_status()

print('Health:', health_response.json())
print('Operations:', [item['name'] for item in operations_response.json()])
print('Pipelines:', [item['name'] for item in pipelines_response.json()])

## 2. 입력 이미지 로드

`IMAGE_PATH`가 지정되면 파일을 읽어 업로드합니다. `None`이면 API 기능을 바로 시험할 수 있도록 경계·선·문자가 들어간 데모 이미지를 생성합니다. 현재 Canny 입력 계약에 맞춰 `FORCE_GRAYSCALE=True`일 때 컬러 입력만 grayscale PNG로 변환합니다. 실제 SEM 이미지에서는 한글/공백이 포함된 Windows 경로도 `Path(r'...')` 형태로 지정할 수 있습니다.

In [ ]:
def create_demo_image() -> np.ndarray:
    image = np.zeros((480, 640, 3), dtype=np.uint8)
    gradient = np.linspace(25, 190, image.shape[1], dtype=np.uint8)
    image[:] = gradient[np.newaxis, :, np.newaxis]
    cv2.rectangle(image, (70, 80), (280, 300), (240, 240, 240), -1)
    cv2.circle(image, (455, 210), 105, (45, 210, 80), -1)
    cv2.line(image, (40, 400), (600, 340), (255, 255, 255), 8)
    cv2.putText(
        image, 'API TEST', (190, 445),
        cv2.FONT_HERSHEY_SIMPLEX, 1.3, (10, 10, 10), 3, cv2.LINE_AA,
    )
    return image


def load_input_image(
    image_path: Path | None,
) -> tuple[np.ndarray, bytes, str, str]:
    if image_path is None:
        image = create_demo_image()
        encoded_ok, encoded = cv2.imencode('.png', image)
        if not encoded_ok:
            raise RuntimeError('데모 이미지를 PNG로 인코딩하지 못했습니다.')
        return image, encoded.tobytes(), 'demo_input.png', 'image/png'

    resolved = image_path.expanduser().resolve()
    if not resolved.is_file():
        raise FileNotFoundError(f'이미지 파일을 찾을 수 없습니다: {resolved}')

    encoded_bytes = resolved.read_bytes()
    encoded_array = np.frombuffer(encoded_bytes, dtype=np.uint8)
    image = cv2.imdecode(encoded_array, cv2.IMREAD_UNCHANGED)
    if image is None:
        raise ValueError(f'OpenCV가 이미지를 디코딩하지 못했습니다: {resolved}')

    media_type = mimetypes.guess_type(resolved.name)[0] or 'image/png'
    return image, encoded_bytes, resolved.name, media_type


input_image, input_bytes, input_filename, input_media_type = (
    load_input_image(IMAGE_PATH)
)

if FORCE_GRAYSCALE and input_image.ndim == 3:
    if input_image.shape[2] == 3:
        input_image = cv2.cvtColor(input_image, cv2.COLOR_BGR2GRAY)
    elif input_image.shape[2] == 4:
        input_image = cv2.cvtColor(input_image, cv2.COLOR_BGRA2GRAY)
    else:
        raise ValueError(f'지원하지 않는 채널 수입니다: {input_image.shape}')

    encoded_ok, encoded = cv2.imencode('.png', input_image)
    if not encoded_ok:
        raise RuntimeError('grayscale 이미지를 PNG로 인코딩하지 못했습니다.')
    input_bytes = encoded.tobytes()
    input_filename = f'{Path(input_filename).stem}_gray.png'
    input_media_type = 'image/png'

print(
    f'Input: {input_filename} | shape={input_image.shape} | '
    f'dtype={input_image.dtype} | encoded={len(input_bytes):,} bytes'
)

In [ ]:
def to_display_image(image: np.ndarray) -> tuple[np.ndarray, str | None]:
    if image.ndim == 2:
        return image, 'gray'
    if image.shape[2] == 3:
        return cv2.cvtColor(image, cv2.COLOR_BGR2RGB), None
    if image.shape[2] == 4:
        return cv2.cvtColor(image, cv2.COLOR_BGRA2RGBA), None
    raise ValueError(f'표시할 수 없는 이미지 shape입니다: {image.shape}')


def show_images(images: dict[str, np.ndarray], columns: int = 3) -> None:
    rows = int(np.ceil(len(images) / columns))
    figure, axes = plt.subplots(
        rows, columns, figsize=(5 * columns, 4 * rows), squeeze=False
    )
    flat_axes = axes.ravel()

    for axis, (title, image) in zip(flat_axes, images.items(), strict=False):
        displayed, color_map = to_display_image(image)
        axis.imshow(displayed, cmap=color_map, vmin=0, vmax=255)
        axis.set_title(f'{title}\nshape={image.shape}, dtype={image.dtype}')
        axis.axis('off')

    for axis in flat_axes[len(images):]:
        axis.axis('off')

    figure.tight_layout()
    plt.show()


show_images({'Original': input_image}, columns=1)

## 3. API 호출 및 결과 디코딩 함수

이미지는 `multipart/form-data`의 `files`에 넣고, 입력 binding과 파라미터는 `payload` JSON 문자열로 전달합니다. 서버 오류가 발생하면 상태 코드와 구조화된 오류 응답을 함께 표시합니다.

In [ ]:
def raise_for_api_error(response: httpx.Response) -> None:
    if response.is_success:
        return
    try:
        detail: Any = response.json()
    except ValueError:
        detail = response.text
    raise RuntimeError(
        f'API 요청 실패: HTTP {response.status_code}\n'
        f'{json.dumps(detail, ensure_ascii=False, indent=2)}'
    )


def decode_result_image(
    result: dict[str, Any],
    output_name: str,
) -> np.ndarray:
    image_info = result['images'][output_name]
    encoded = base64.b64decode(image_info['data'], validate=True)
    array = np.frombuffer(encoded, dtype=np.uint8)

    if image_info['media_type'] == 'image/png':
        decoded = cv2.imdecode(array, cv2.IMREAD_UNCHANGED)
        if decoded is None:
            raise RuntimeError(f'출력 이미지를 디코딩하지 못했습니다: {output_name}')
        return decoded

    raise ValueError(
        f"Notebook 예제는 PNG 출력만 표시합니다: {image_info['media_type']}"
    )


def execute_operation(
    operation: str,
    params: dict[str, Any] | None = None,
) -> dict[str, Any]:
    payload = {
        'params': params or {},
        'image_inputs': [{'input_name': 'image', 'file_index': 0}],
    }
    response = client.post(
        f'{API_PREFIX}/operations/{operation}/execute',
        data={'payload': json.dumps(payload)},
        files={
            'files': (input_filename, input_bytes, input_media_type),
        },
    )
    raise_for_api_error(response)
    return response.json()


def execute_pipeline(
    pipeline_name: str,
    *,
    retain_intermediates: bool = True,
) -> dict[str, Any]:
    payload = {
        'retain_intermediates': retain_intermediates,
        'image_inputs': [{'input_name': 'image', 'file_index': 0}],
    }
    response = client.post(
        f'{API_PREFIX}/pipelines/{pipeline_name}/execute',
        data={'payload': json.dumps(payload)},
        files={
            'files': (input_filename, input_bytes, input_media_type),
        },
    )
    raise_for_api_error(response)
    return response.json()

## 4. 개별 전처리 Operation 비교

각 파라미터를 바꾼 뒤 셀을 다시 실행하면 결과 차이를 바로 비교할 수 있습니다. `kernel_size`는 홀수여야 하고 Canny는 `threshold_low < threshold_high`여야 합니다.

In [ ]:
blur_body = execute_operation(
    'gaussian_blur',
    params={'kernel_size': 9, 'sigma_x': 1.5},
)
canny_body = execute_operation(
    'canny',
    params={'threshold_low': 70, 'threshold_high': 160},
)
resize_body = execute_operation(
    'resize',
    params={'width': 320, 'height': 240, 'interpolation': 'area'},
)

blurred = decode_result_image(blur_body['output'], 'image')
edges = decode_result_image(canny_body['output'], 'image')
resized = decode_result_image(resize_body['output'], 'image')

show_images(
    {
        'Original': input_image,
        'Gaussian Blur': blurred,
        'Canny Edge': edges,
        'Resize': resized,
    },
    columns=2,
)

In [ ]:
print('Gaussian Blur metadata:')
print(json.dumps(blur_body['metadata'], ensure_ascii=False, indent=2))
print('\nCanny metadata:')
print(json.dumps(canny_body['metadata'], ensure_ascii=False, indent=2))

## 5. 등록 Pipeline과 중간 결과 확인

기본 `edge_thumbnail` Pipeline은 Color 변환 → Gaussian Blur → Resize → Canny 순서로 실행됩니다. `retain_intermediates=True`이므로 각 단계의 출력도 함께 표시합니다. 운영 환경에서 중간 결과가 필요 없다면 `False`로 두는 편이 응답 크기와 메모리 사용량에 유리합니다.

In [ ]:
pipeline_body = execute_pipeline(
    'edge_thumbnail',
    retain_intermediates=True,
)

pipeline_images: dict[str, np.ndarray] = {'Original': input_image}
for step_id, step_output in pipeline_body['intermediates'].items():
    for output_name in step_output['images']:
        title = f'{step_id}.{output_name}'
        pipeline_images[title] = decode_result_image(step_output, output_name)

for output_name in pipeline_body['output']['images']:
    pipeline_images[f'Final.{output_name}'] = decode_result_image(
        pipeline_body['output'], output_name
    )

show_images(pipeline_images, columns=3)

print('Step results:')
for step in pipeline_body['steps']:
    print(
        f"- {step['step_id']}: {step['operation']} | "
        f"success={step['success']} | "
        f"{step['metadata']['duration_ms']:.3f} ms"
    )

## 6. 선택: 결과 이미지 저장

필요할 때만 아래 셀의 `SAVE_RESULTS`를 `True`로 바꾸세요. Notebook 실행 때마다 디스크 파일을 만들지 않도록 기본값은 `False`입니다.

In [ ]:
SAVE_RESULTS = False
OUTPUT_DIR = Path('notebook_outputs')

if SAVE_RESULTS:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    for name, image in {
        'gaussian_blur.png': blurred,
        'canny.png': edges,
        'resize.png': resized,
        'pipeline_edges.png': pipeline_images['Final.edges'],
    }.items():
        output_path = OUTPUT_DIR / name
        if not cv2.imwrite(str(output_path), image):
            raise RuntimeError(f'이미지 저장 실패: {output_path}')
        print(f'Saved: {output_path.resolve()}')

## 마무리

테스트가 끝나면 연결을 닫습니다. 서버는 실행한 터미널에서 `Ctrl+C`로 종료할 수 있습니다.

In [ ]:
client.close()